In [3]:
data = """call_id,caller,receiver,city,call_type,duration_seconds,cost
C001,Amit,Rahul,Hyderabad,Local,180,2.5
C002,Neha,Arjun,Bangalore,STD,320,6.0
C003,Rahul,Pooja,Delhi,Local,60,1.0
C004,Pooja,Neha,Mumbai,ISD,900,25.0
C005,Arjun,Amit,Chennai,STD,400,7.5
C006,Sneha,Karan,Hyderabad,Local,240,3.0
C007,Karan,Sneha,Delhi,Local,120,2.0
C008,Riya,Vikas,Bangalore,STD,360,6.5
C009,Vikas,Riya,Mumbai,ISD,1100,30.0
C010,Anjali,Sanjay,Chennai,Local,90,1.5
C011,Farhan,Ayesha,Delhi,STD,420,7.0
C012,Ayesha,Farhan,Hyderabad,ISD,950,28.0
C013,Suresh,Divya,Bangalore,Local,150,2.0
C014,Divya,Suresh,Mumbai,STD,380,6.8
C015,Nikhil,Priya,Delhi,Local,200,2.8
C016,Priya,Nikhil,Chennai,STD,410,7.2
C017,Rohit,Kavya,Hyderabad,Local,170,2.3
C018,Kavya,Rohit,Bangalore,Local,140,2.1
C019,Manish,Tina,Mumbai,ISD,1000,27.0
C020,Tina,Manish,Delhi,STD,350,6.2
"""

with open("/content/call_records.csv", "w") as f:
    f.write(data)


In [4]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

rdd = sc.textFile("/content/call_records.csv")

header = rdd.first()
data_rdd = rdd.filter(lambda x: x != header)

records = data_rdd.map(lambda x: x.split(",")) \
    .map(lambda x: (
        x[0],          # call_id
        x[1],          # caller
        x[2],          # receiver
        x[3],          # city
        x[4],          # call_type
        int(x[5]),     # duration_seconds
        float(x[6])    # cost
    ))

In [5]:
records.take(5)

[('C001', 'Amit', 'Rahul', 'Hyderabad', 'Local', 180, 2.5),
 ('C002', 'Neha', 'Arjun', 'Bangalore', 'STD', 320, 6.0),
 ('C003', 'Rahul', 'Pooja', 'Delhi', 'Local', 60, 1.0),
 ('C004', 'Pooja', 'Neha', 'Mumbai', 'ISD', 900, 25.0),
 ('C005', 'Arjun', 'Amit', 'Chennai', 'STD', 400, 7.5)]

In [6]:
records.map(lambda x: (x[3],x[6]))\
  .reduceByKey(lambda a,b:a+b)\
  .collect()

[('Hyderabad', 35.8),
 ('Delhi', 19.0),
 ('Mumbai', 88.8),
 ('Bangalore', 16.6),
 ('Chennai', 16.2)]

In [7]:
records.map(lambda x: (x[3],x[6]))\
  .reduceByKey(lambda a,b:a+b)\
  .max(key=lambda x:x[1])

('Mumbai', 88.8)

In [8]:
records.map(lambda x: (x[4],x[5]))\
  .reduceByKey(lambda a,b:a+b)\
  .collect()

[('Local', 1350), ('STD', 2640), ('ISD', 3950)]

In [9]:
records.map(lambda x: (x[3],1))\
  .reduceByKey(lambda a,b:a+b)\
  .collect()

[('Hyderabad', 4),
 ('Delhi', 5),
 ('Mumbai', 4),
 ('Bangalore', 4),
 ('Chennai', 3)]

In [10]:
records.map(lambda x: (x[3],(x[6],1)))\
  .reduceByKey(lambda a,b:(a[0]+b[0],a[1]+b[1]))\
  .mapValues(lambda x:x[0]/x[1])\
  .collect()

[('Hyderabad', 8.95),
 ('Delhi', 3.8),
 ('Mumbai', 22.2),
 ('Bangalore', 4.15),
 ('Chennai', 5.3999999999999995)]

In [12]:
records.filter(lambda x:x[6]>20).collect()

[('C004', 'Pooja', 'Neha', 'Mumbai', 'ISD', 900, 25.0),
 ('C009', 'Vikas', 'Riya', 'Mumbai', 'ISD', 1100, 30.0),
 ('C012', 'Ayesha', 'Farhan', 'Hyderabad', 'ISD', 950, 28.0),
 ('C019', 'Manish', 'Tina', 'Mumbai', 'ISD', 1000, 27.0)]

In [13]:
from types import LambdaType
records.filter(lambda x:x[4]=='ISD')\
   .map(lambda x:(x[3],1))\
   .reduceByKey(lambda a,b:a+b)\
   .collect()

[('Mumbai', 3), ('Hyderabad', 1)]

In [14]:
records.max(key=lambda x:x[5])

('C009', 'Vikas', 'Riya', 'Mumbai', 'ISD', 1100, 30.0)

In [17]:
records.map(lambda x:(x[1],x[6]))\
  .reduceByKey(lambda a,b:a+b)\
  .collect()


[('Amit', 2.5),
 ('Pooja', 25.0),
 ('Karan', 2.0),
 ('Riya', 6.5),
 ('Vikas', 30.0),
 ('Suresh', 2.0),
 ('Divya', 6.8),
 ('Nikhil', 2.8),
 ('Rohit', 2.3),
 ('Manish', 27.0),
 ('Tina', 6.2),
 ('Neha', 6.0),
 ('Rahul', 1.0),
 ('Arjun', 7.5),
 ('Sneha', 3.0),
 ('Anjali', 1.5),
 ('Farhan', 7.0),
 ('Ayesha', 28.0),
 ('Priya', 7.2),
 ('Kavya', 2.1)]

In [18]:
records.filter(lambda x:x[5]>900 and x[6]>25).collect()

[('C009', 'Vikas', 'Riya', 'Mumbai', 'ISD', 1100, 30.0),
 ('C012', 'Ayesha', 'Farhan', 'Hyderabad', 'ISD', 950, 28.0),
 ('C019', 'Manish', 'Tina', 'Mumbai', 'ISD', 1000, 27.0)]